# Flight Delay Prediction - External Invocation Workbook

Este workbook representa a una persona externa que no entrena el modelo ni accede al archivo `.joblib`. Su unica responsabilidad es invocar la API publicada con ngrok y leer la estimacion de minutos de retraso.

Uso esperado:

1. Ejecutar primero `Flight Delay API Ngrok Demo.ipynb`.
2. Copiar la URL publica de ngrok.
3. Pegarla en `NGROK_URL`.
4. Ejecutar este workbook para consumir el modelo remotamente.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "requests", "-q"])

import requests

print("requests listo")

## 1. Configurar URL publica

Reemplaza el valor por la URL que imprimio ngrok. Debe verse similar a `https://xxxxx.ngrok-free.app`.

In [ ]:
NGROK_URL = "https://REEMPLAZA-CON-TU-URL.ngrok-free.app"

NGROK_URL = NGROK_URL.rstrip("/")

if "REEMPLAZA-CON-TU-URL" in NGROK_URL:
    raise ValueError("Pega aqui la URL publica real de ngrok antes de continuar.")

print("API configurada:", NGROK_URL)

## 2. Verificar disponibilidad de la API

In [ ]:
health = requests.get(f"{NGROK_URL}/health", timeout=15)
print("Status code:", health.status_code)
print(health.json())

## 3. Invocar prediccion

Este ejemplo envia las features esperadas por el modelo. La respuesta incluye minutos estimados de retraso de llegada y nivel de riesgo operativo.

In [ ]:
flight_payload = {
    "features": {
        "HOUR_OF_DAY": 18,
        "IS_PEAK_HOUR": 1,
        "IS_WEEKEND": 1,
        "IS_HIGH_SEASON": 1,
        "CASCADING_DELAY_FLAG": 1,
        "ROUTE_AVG_DELAY": 18.5,
        "AIRLINE_AVG_DELAY": 11.2,
        "AIRLINE_ENC": 5,
        "DISTANCE": 1189,
        "DEPARTURE_DELAY": 22,
        "MONTH": 12,
        "DAY_OF_WEEK": 7,
        "AIR_SYSTEM_DELAY": 12,
        "WEATHER_DELAY": 6,
    }
}

response = requests.post(f"{NGROK_URL}/predict", json=flight_payload, timeout=15)
print("Status code:", response.status_code)

prediction = response.json()
prediction

## 4. Interpretacion para negocio

In [ ]:
predicted_minutes = prediction["predicted_arrival_delay_minutes"]
risk_level = prediction["risk_level"]

print(f"Retraso estimado de llegada: {predicted_minutes:.2f} minutos")
print(f"Nivel de riesgo: {risk_level}")

if predicted_minutes >= 45:
    print("Decision sugerida: activar alerta operativa critica y preparar comunicacion proactiva.")
elif predicted_minutes > 15:
    print("Decision sugerida: monitorear como riesgo medio y preparar recursos preventivos.")
else:
    print("Decision sugerida: monitorear sin alerta critica por ahora.")